[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/078.%20The%20Fleet%20Sizing%20and%20Composition%20Problem/P78-Tier-7_executed.ipynb)

---


# 78. The Fleet Sizing and Composition Problem
## Tier 7: The Human–AI Symbiotic Partnership Method

### Purpose and scope

This notebook demonstrates a human–AI decision workflow for a small fleet-sizing example. An explainable AI component proposes an allocation, a human-expertise model adjusts that proposal, and a collaborative engine combines the two. A trust system tracks interaction outcomes, while a feedback component records patterns associated with successful decisions.

The implementation is an illustrative simulation, not a fleet optimizer or a validated operational decision-support system. The AI allocation is a fixed rule-based proposal with small context-dependent adjustments; it does not solve a mathematical optimization problem. The performance values are simulated indicators, not observed business outcomes. In particular, recording feedback does not retrain the allocation policy. These distinctions are important when interpreting the plots and metrics.

The example contains three vehicle types, two routes, four expert profiles, and a sequence of simulated interactions. Vehicle allocations are allowed to be fractional because the human–AI synthesis is a weighted average. They therefore represent a decision score or planning signal, not a directly dispatchable integer fleet. The model treats each vehicle-type/route allocation as a separate assignment and charges acquisition cost for each assigned unit. It does not model vehicle reuse across routes, driver availability, time windows, financing, or stochastic service outcomes.

### Method and mathematical representation

For each interaction, the AI first proposes an allocation matrix \(A^{AI}\), and the selected expert produces an adjusted matrix \(A^{H}\). The collaborative allocation is their trust- and confidence-weighted average:

$$
A^{C} = \alpha A^{AI} + (1-\alpha) A^{H},
\qquad
\alpha = \frac{T}{T+C_H}.
$$

Here, \(T\in[0,1]\) is the trust score before the current interaction, \(C_H\in[0,1]\) is the human decision confidence, and \(A^C\) is the final allocation. The code implements this calculation in `_synthesize_decision`. When trust is higher, the AI proposal receives more weight; when human confidence is higher relative to trust, the expert's proposal receives more weight.

Trust begins at \(T_0=0.5\). For interaction \(k\), trust is updated from simulated performance, explanation length, AI confidence, and change in agreement:

$$
T_{k+1} = \operatorname{clip}_{[0,1]}\left(T_k +
w_p\Delta p_k + w_x\Delta x_k + w_r\Delta r_k + w_l\Delta l_k\right),
$$

where the weights are \(w_p=0.4\), \(w_x=0.3\), \(w_r=0.2\), and \(w_l=0.1\). The impacts Δp, Δx, Δr, and Δl are defined in `calculate_trust_score`; clipping keeps trust between zero and one. The update is a pedagogical scoring rule, not an empirically calibrated model of human trust.

The simulated performance score combines service coverage and a normalized cost score:

$$
P = \operatorname{clip}_{[0,1]}\left(0.6S + 0.4C + \varepsilon\right),
\qquad
C = \max\left(0, 1-\frac{\text{cost}-320000}{320000}\right),
$$

where \(S\) is the average route service level and \(\varepsilon\) is simulated noise. The reference value in the cost score is a fixed normalization constant in this demonstration; it is not an independently established optimum. Since interactions are simulated sequentially and update trust and feedback records, outcomes depend on the random context and noise. No convergence or optimality guarantee is claimed.

### Workflow

1. Define the vehicle, route, expert, and decision data structures.
2. Generate an explainable AI proposal and model expert adjustments.
3. Synthesize a collaborative allocation and calculate agreement and simulated performance.
4. Update trust and record feedback for each interaction.
5. Visualize the interaction history and summarize the resulting metrics.

The principal outputs are allocation and cost summaries, interaction-level trust, agreement and performance scores, plots, and qualitative insights. These outputs describe this simulation only; they should not be interpreted as evidence that collaboration outperforms an independently measured AI-only baseline.

In [ ]:
from dataclasses import dataclass
from datetime import datetime
from typing import Any, Dict, List, Optional, Tuple
import random

import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8")

print("Human–AI partnership simulation components are ready.")

In [ ]:
@dataclass
class VehicleType:
    """Describe one vehicle type and its modeled characteristics."""

    id: int
    name: str
    capacity: int
    acquisition_cost: float
    operating_costs: Dict[int, float]
    service_rate: float
    reliability: float
    flexibility: float


@dataclass
class Route:
    """Describe a route and its demand characteristics."""

    id: int
    name: str
    demand_units: int
    arrival_rate: float
    max_delay_prob: float = 0.1
    complexity: float = 1.0
    uncertainty: float = 0.1


@dataclass
class HumanExpert:
    """Describe an expert's area, style, and individual attributes."""

    id: int
    name: str
    expertise_area: str
    experience_years: int
    decision_style: str
    confidence_level: float
    risk_tolerance: float
    learning_rate: float


@dataclass
class AIDecision:
    """Store an AI proposal, its scores, explanations, and alternatives."""

    fleet_allocation: np.ndarray
    total_cost: float
    service_level: float
    confidence: float
    reasoning: List[str]
    alternatives: List[Tuple[np.ndarray, str]]
    uncertainty_factors: List[str]


@dataclass
class HumanDecision:
    """Store an expert's adjusted proposal and supporting rationale."""

    expert_id: int
    fleet_allocation: np.ndarray
    rationale: str
    confidence: float
    intuition_factors: List[str]
    concerns: List[str]


@dataclass
class CollaborativeDecision:
    """Record the decisions and measured indicators for one interaction."""

    timestamp: datetime
    ai_decision: AIDecision
    human_input: Optional[HumanDecision]
    final_allocation: np.ndarray
    trust_score: float
    agreement_level: float
    learning_outcome: str
    performance_score: float


@dataclass
class PartnershipResult:
    """Collect the final simulation outputs and summary metrics."""

    optimal_fleet: Dict[Tuple[int, int], float]
    total_cost: float
    trust_development: List[float]
    decision_quality_improvement: float
    learning_effectiveness: float
    partnership_benefits: Dict[str, float]
    computation_time: float
    insights: List[str]

In [ ]:
class ExplainableAI:
    """Produce rule-based fleet proposals with human-readable explanations."""

    def __init__(self, vehicles: List[VehicleType], routes: List[Route]):
        self.vehicles = vehicles
        self.routes = routes
        self.num_vehicles = len(vehicles)
        self.num_routes = len(routes)
        self.human_preferences: Dict[int, Dict[str, Any]] = {}
        self.success_patterns: List[Dict[str, Any]] = []
        self.explanation_templates = self._initialize_explanations()
        self.optimization_engine = self._create_optimization_engine()
        self.reasoning_engine = self._create_reasoning_engine()

        print("Explainable AI component initialized:")
        print(f"  Vehicle types: {self.num_vehicles}")
        print(f"  Routes: {self.num_routes}")
        print(f"  Explanation categories: {len(self.explanation_templates)}")

    def _initialize_explanations(self) -> Dict[str, List[str]]:
        """Provide reusable explanation text for decision contexts."""
        return {
            "cost_optimization": [
                "The proposal emphasizes combinations with lower modeled costs.",
                "Acquisition and route operating costs are included in the cost score.",
                "The allocation is presented as a cost-focused rule-based proposal.",
            ],
            "service_level": [
                "Route capacity is compared with the demand values in the context.",
                "The service score is capped at full demand coverage for each route.",
                "Demand variation can change the estimated service score.",
            ],
            "risk_management": [
                "Vehicle reliability is available as a modeled characteristic.",
                "The demonstration does not calculate a probabilistic risk measure.",
                "Unmodeled disruptions remain a limitation of this example.",
            ],
            "operational_efficiency": [
                "The proposal assigns vehicle types to individual routes.",
                "The example does not model driver schedules or vehicle reuse.",
                "Operational characteristics are descriptive rather than optimized.",
            ],
        }

    def _create_optimization_engine(self) -> Dict[str, Any]:
        """Describe the illustrative proposal mechanism without implying optimization."""
        return {
            "method": "fixed_rule_based_proposal",
            "parameters": {"context_adjustments": True},
        }

    def _create_reasoning_engine(self) -> Dict[str, float]:
        """Store the example's stated emphasis weights for reporting."""
        return {
            "cost_weight": 0.4,
            "service_weight": 0.3,
            "risk_weight": 0.2,
            "operational_weight": 0.1,
        }

    def generate_decision(self, context: Dict[str, Any]) -> AIDecision:
        """Generate a proposal, its metrics, and its explanation."""
        allocation = self._optimize_fleet(context)
        demands = np.asarray(
            context.get("demands", [route.demand_units for route in self.routes]),
            dtype=float,
        )
        total_cost = self._calculate_total_cost(allocation)
        service_level = self._calculate_service_level(allocation, demands)
        confidence = self._calculate_confidence(context)
        reasoning = self._generate_reasoning(allocation, context)
        alternatives = self._generate_alternatives(allocation)
        uncertainties = self._identify_uncertainties(context)

        return AIDecision(
            fleet_allocation=allocation,
            total_cost=total_cost,
            service_level=service_level,
            confidence=confidence,
            reasoning=reasoning,
            alternatives=alternatives,
            uncertainty_factors=uncertainties,
        )

    def _optimize_fleet(self, context: Dict[str, Any]) -> np.ndarray:
        """Return the demonstration's fixed proposal with context adjustments."""
        allocation = np.array(
            [[2, 1], [1, 2], [0, 1]],
            dtype=float,
        )
        if context.get("cost_focus", False):
            allocation[0] += 1
        elif context.get("service_focus", False):
            allocation[2] += 1
        return allocation

    def _calculate_total_cost(self, fleet_allocation: np.ndarray) -> float:
        """Charge acquisition and route operating cost per assigned unit."""
        total_cost = 0.0
        for i, vehicle in enumerate(self.vehicles):
            for j, route in enumerate(self.routes):
                count = fleet_allocation[i, j]
                total_cost += count * (
                    vehicle.acquisition_cost + vehicle.operating_costs[route.id]
                )
        return float(total_cost)

    def _calculate_service_level(
        self,
        fleet_allocation: np.ndarray,
        demands: np.ndarray,
    ) -> float:
        """Return mean route coverage, capped at one for each route."""
        if len(demands) != self.num_routes:
            raise ValueError("The demand vector must contain one value per route.")
        service_levels = []
        for j in range(self.num_routes):
            capacity = sum(
                fleet_allocation[i, j] * vehicle.capacity
                for i, vehicle in enumerate(self.vehicles)
            )
            demand = demands[j]
            service_levels.append(min(1.0, capacity / demand) if demand > 0 else 1.0)
        return float(np.mean(service_levels))

    def _calculate_confidence(self, context: Dict[str, Any]) -> float:
        """Reduce baseline confidence as the context's uncertainty increases."""
        uncertainty = float(context.get("uncertainty", 0.1))
        return float(np.clip(0.8 * (1.0 - uncertainty), 0.5, 1.0))

    def _generate_reasoning(
        self,
        fleet_allocation: np.ndarray,
        context: Dict[str, Any],
    ) -> List[str]:
        """Select explanations relevant to the proposal's context."""
        reasoning = []
        if context.get("cost_focus", False):
            reasoning.extend(
                random.sample(self.explanation_templates["cost_optimization"], 2)
            )
        if context.get("service_focus", False):
            reasoning.extend(
                random.sample(self.explanation_templates["service_level"], 2)
            )
        reasoning.extend(
            random.sample(self.explanation_templates["operational_efficiency"], 1)
        )
        reasoning.append(
            f"The proposal contains {fleet_allocation.sum():.1f} assigned vehicle units."
        )
        return reasoning

    def _generate_alternatives(
        self,
        fleet_allocation: np.ndarray,
    ) -> List[Tuple[np.ndarray, str]]:
        """Create illustrative neighboring proposals for discussion."""
        cost_alt = fleet_allocation.copy()
        cost_alt[0] += 1
        cost_alt[2] = np.maximum(0, cost_alt[2] - 0.5)

        service_alt = fleet_allocation.copy()
        service_alt[2] += 1
        service_alt[0] = np.maximum(0, service_alt[0] - 0.5)
        return [
            (cost_alt, "Illustrative alternative with more small-vehicle units."),
            (service_alt, "Illustrative alternative with more large-vehicle units."),
        ]

    def _identify_uncertainties(self, context: Dict[str, Any]) -> List[str]:
        """List modeled and unmodeled sources of uncertainty."""
        uncertainties = []
        if context.get("demand_uncertainty", 0) > 0.1:
            uncertainties.append("Demand uncertainty may affect service estimates.")
        if context.get("cost_volatility", 0) > 0.1:
            uncertainties.append("Operating-cost volatility is not quantified here.")
        uncertainties.extend(
            [
                "Weather conditions are not represented in the allocation rules.",
                "Maintenance-related vehicle availability is not modeled.",
            ]
        )
        return uncertainties

    def learn_from_feedback(
        self,
        human_decision: HumanDecision,
        performance: float,
        expert: HumanExpert,
    ) -> None:
        """Record feedback patterns; this demonstration does not retrain policy."""
        if performance > 0.8:
            self.success_patterns.append(
                {
                    "allocation": human_decision.fleet_allocation.copy(),
                    "rationale": human_decision.rationale,
                    "performance": performance,
                    "expertise_area": expert.expertise_area,
                }
            )

        expert_id = human_decision.expert_id
        if expert_id not in self.human_preferences:
            self.human_preferences[expert_id] = {
                "preferred_patterns": [],
                "decision_style": expert.decision_style,
                "success_rate": performance,
                "observations": 1,
            }
        else:
            preferences = self.human_preferences[expert_id]
            preferences["preferred_patterns"].append(
                human_decision.fleet_allocation.copy()
            )
            preferences["success_rate"] = (
                0.7 * preferences["success_rate"] + 0.3 * performance
            )
            preferences["observations"] += 1

In [ ]:
class HumanExpertiseModel:
    """Generate expert adjustments and explanations from profile attributes."""

    def __init__(self, experts: List[HumanExpert]):
        self.experts = experts
        self.expertise_patterns = self._initialize_expertise_patterns()
        self.decision_styles = self._initialize_decision_styles()
        print(f"Human expertise model initialized for {len(experts)} experts.")

    def _initialize_expertise_patterns(self) -> Dict[str, Dict[str, Any]]:
        """Define the metrics and considerations associated with each specialty."""
        return {
            "cost": {
                "focus_metrics": ["total_cost", "cost_efficiency", "return"],
                "decision_factors": ["acquisition_cost", "operating_cost"],
                "risk_tolerance": 0.3,
                "typical_decisions": "cost_minimization",
            },
            "service": {
                "focus_metrics": ["service_level", "reliability"],
                "decision_factors": ["capacity", "coverage", "response_time"],
                "risk_tolerance": 0.7,
                "typical_decisions": "service_maximization",
            },
            "risk": {
                "focus_metrics": ["risk_exposure", "reliability"],
                "decision_factors": ["reliability", "flexibility", "redundancy"],
                "risk_tolerance": 0.2,
                "typical_decisions": "risk_mitigation",
            },
            "operations": {
                "focus_metrics": ["efficiency", "utilization", "productivity"],
                "decision_factors": ["maintenance", "scheduling", "workflows"],
                "risk_tolerance": 0.5,
                "typical_decisions": "operational_optimization",
            },
        }

    def _initialize_decision_styles(self) -> Dict[str, Dict[str, Any]]:
        """Define illustrative confidence and AI-trust values for each style."""
        return {
            "analytical": {
                "approach": "data_driven",
                "confidence_base": 0.8,
                "reasoning_style": "quantitative",
                "ai_trust": 0.9,
            },
            "intuitive": {
                "approach": "experience_based",
                "confidence_base": 0.7,
                "reasoning_style": "qualitative",
                "ai_trust": 0.6,
            },
            "collaborative": {
                "approach": "team_oriented",
                "confidence_base": 0.75,
                "reasoning_style": "integrative",
                "ai_trust": 0.8,
            },
            "cautious": {
                "approach": "risk_averse",
                "confidence_base": 0.6,
                "reasoning_style": "conservative",
                "ai_trust": 0.7,
            },
        }

    def get_human_decision(
        self,
        expert_id: int,
        ai_decision: AIDecision,
        context: Dict[str, Any],
    ) -> HumanDecision:
        """Apply one expert profile to an AI proposal."""
        expert = next((item for item in self.experts if item.id == expert_id), None)
        if expert is None:
            raise ValueError(f"No expert has ID {expert_id}.")

        pattern = self.expertise_patterns[expert.expertise_area]
        style = self.decision_styles[expert.decision_style]
        allocation = self._evaluate_ai_decision(ai_decision, expert, pattern)
        rationale = self._generate_human_rationale(expert, style)
        confidence = self._calculate_human_confidence(ai_decision, expert, style)
        intuition = self._identify_intuition_factors(expert, context)
        concerns = self._identify_concerns(ai_decision, expert)

        return HumanDecision(
            expert_id=expert_id,
            fleet_allocation=allocation,
            rationale=rationale,
            confidence=confidence,
            intuition_factors=intuition,
            concerns=concerns,
        )

    def _evaluate_ai_decision(
        self,
        ai_decision: AIDecision,
        expert: HumanExpert,
        expertise_pattern: Dict[str, Any],
    ) -> np.ndarray:
        """Adjust the AI allocation in the direction of the expert's specialty."""
        allocation = ai_decision.fleet_allocation.copy()
        if expert.expertise_area == "cost":
            allocation[0] += 0.5
            allocation[2] -= 0.5
        elif expert.expertise_area == "service":
            allocation[2] += 0.5
            allocation[0] -= 0.5
        elif expert.expertise_area == "risk":
            allocation[1] += 0.5
        elif expert.expertise_area == "operations":
            allocation[0] += 0.3
            allocation[1] += 0.2
        return np.maximum(allocation, 0.0)

    def _generate_human_rationale(
        self,
        expert: HumanExpert,
        decision_style: Dict[str, Any],
    ) -> str:
        """Summarize the expert's specialty and decision style."""
        rationale_by_area = {
            "cost": "The cost-focused adjustment emphasizes the modeled acquisition and operating costs.",
            "service": "The service-focused adjustment emphasizes route capacity and demand coverage.",
            "risk": "The risk-focused adjustment emphasizes a more balanced vehicle mix.",
            "operations": "The operations-focused adjustment emphasizes a flexible vehicle allocation.",
        }
        style_notes = {
            "analytical": "The recommendation is framed as a quantitative assessment.",
            "intuitive": "The recommendation is framed around experience-based judgment.",
            "collaborative": "The recommendation is framed as balancing stakeholder needs.",
            "cautious": "The recommendation is framed with attention to potential risks.",
        }
        trust_note = (
            "The expert gives the AI proposal substantial consideration."
            if decision_style["ai_trust"] > 0.7
            else "The expert treats the AI proposal as one input among several."
        )
        return " ".join(
            [rationale_by_area[expert.expertise_area], style_notes[expert.decision_style], trust_note]
        )

    def _calculate_human_confidence(
        self,
        ai_decision: AIDecision,
        expert: HumanExpert,
        decision_style: Dict[str, Any],
    ) -> float:
        """Combine style-based confidence, AI trust, and experience."""
        confidence = (
            decision_style["confidence_base"]
            + decision_style["ai_trust"] * ai_decision.confidence * 0.1
            + expert.experience_years / 50 * 0.1
        )
        return float(np.clip(confidence, 0.3, 1.0))

    def _identify_intuition_factors(
        self,
        expert: HumanExpert,
        context: Dict[str, Any],
    ) -> List[str]:
        """List contextual considerations associated with the expert's specialty."""
        factors_by_area = {
            "cost": ["Fuel price trends", "Maintenance cost patterns", "Vehicle depreciation"],
            "service": ["Customer satisfaction", "Seasonal demand", "Service reliability"],
            "risk": ["Weather risks", "Disruptions", "Contingency planning"],
            "operations": ["Driver availability", "Maintenance schedules", "Operational bottlenecks"],
        }
        factors = list(factors_by_area[expert.expertise_area])
        if context.get("market_conditions", "normal") != "normal":
            factors.append("Current market conditions")
        if context.get("seasonal_factors", False):
            factors.append("Seasonal operational patterns")
        return factors

    def _identify_concerns(
        self,
        ai_decision: AIDecision,
        expert: HumanExpert,
    ) -> List[str]:
        """Return example concerns relevant to the expert and AI confidence."""
        concerns_by_area = {
            "cost": ["Long-term costs may be incompletely represented."],
            "service": ["Service quality is represented only through capacity coverage."],
            "risk": ["Rare disruptive events are not modeled."],
            "operations": ["Practical scheduling constraints are not modeled."],
        }
        concerns = list(concerns_by_area[expert.expertise_area])
        if ai_decision.confidence < 0.8:
            concerns.append("The AI confidence score indicates uncertainty.")
        if len(ai_decision.uncertainty_factors) > 3:
            concerns.append("Several uncertainty factors are listed for this context.")
        return concerns

In [ ]:
class TrustDevelopmentSystem:
    """Track a bounded trust score using the simulation's stated scoring rule."""

    def __init__(self):
        self.trust_history: List[float] = []
        self.interaction_history: List[CollaborativeDecision] = []
        self.trust_factors = {
            "performance": 0.4,
            "transparency": 0.3,
            "reliability": 0.2,
            "learning": 0.1,
        }
        print("Trust system initialized with a neutral starting score of 0.5.")

    def calculate_trust_score(self, interaction: CollaborativeDecision) -> float:
        """Calculate the next score without modifying the stored history."""
        base_trust = self.trust_history[-1] if self.trust_history else 0.5
        performance_impact = (interaction.performance_score - 0.7) * 0.2
        transparency_score = min(1.0, len(interaction.ai_decision.reasoning) / 5.0)
        transparency_impact = (transparency_score - 0.5) * 0.1
        reliability_impact = (interaction.ai_decision.confidence - 0.7) * 0.1
        learning_impact = 0.0
        if self.interaction_history:
            previous_agreement = self.interaction_history[-1].agreement_level
            learning_impact = (interaction.agreement_level - previous_agreement) * 0.1

        change = (
            self.trust_factors["performance"] * performance_impact
            + self.trust_factors["transparency"] * transparency_impact
            + self.trust_factors["reliability"] * reliability_impact
            + self.trust_factors["learning"] * learning_impact
        )
        return float(np.clip(base_trust + change, 0.0, 1.0))

    def update_trust(self, interaction: CollaborativeDecision) -> float:
        """Append the new trust score and its associated interaction."""
        score = self.calculate_trust_score(interaction)
        self.trust_history.append(score)
        self.interaction_history.append(interaction)
        return score

    def get_trust_insights(self) -> List[str]:
        """Describe the recent direction and variability of trust scores."""
        if len(self.trust_history) < 5:
            return ["Trust development is based on fewer than five interactions."]

        recent = float(np.mean(self.trust_history[-5:]))
        early = float(np.mean(self.trust_history[:5]))
        trend = recent - early
        if trend > 0.1:
            insights = [f"The recent trust average is higher by {trend:.1%}."]
        elif trend < -0.1:
            insights = [f"The recent trust average is lower by {abs(trend):.1%}."]
        else:
            insights = ["The recent trust average is broadly stable."]

        if recent > 0.8:
            insights.append("The recent average falls in the high-trust range.")
        elif recent > 0.6:
            insights.append("The recent average falls in the moderate-trust range.")
        elif recent > 0.4:
            insights.append("The recent average falls in the neutral range.")
        else:
            insights.append("The recent average falls in the low-trust range.")

        if len(self.trust_history) > 10:
            volatility = float(np.std(self.trust_history[-10:]))
            label = "variable" if volatility > 0.1 else "relatively stable"
            insights.append(f"Recent trust scores are {label} (standard deviation {volatility:.3f}).")
        return insights

    def get_trust_metrics(self) -> Dict[str, float]:
        """Return summary statistics for the recorded trust scores."""
        if not self.trust_history:
            return {
                "current_trust": 0.5,
                "trust_trend": 0.0,
                "trust_volatility": 0.0,
                "peak_trust": 0.5,
                "min_trust": 0.5,
            }
        return {
            "current_trust": self.trust_history[-1],
            "trust_trend": self.trust_history[-1] - self.trust_history[0],
            "trust_volatility": float(np.std(self.trust_history)),
            "peak_trust": max(self.trust_history),
            "min_trust": min(self.trust_history),
        }

In [ ]:
vehicles = [
    VehicleType(1, "Small Truck", 5, 50000, {1: 1000, 2: 1500}, 4.0, 0.85, 0.9),
    VehicleType(2, "Medium Truck", 10, 80000, {1: 1200, 2: 1800}, 3.5, 0.90, 0.7),
    VehicleType(3, "Large Truck", 20, 120000, {1: 1500, 2: 2000}, 3.0, 0.95, 0.5),
]

routes = [
    Route(1, "Urban Route", 15, 8.0, 0.1, 1.2, 0.15),
    Route(2, "Rural Route", 25, 4.0, 0.1, 0.8, 0.10),
]

experts = [
    HumanExpert(1, "Dr. Sarah Chen", "cost", 15, "analytical", 0.85, 0.3, 0.8),
    HumanExpert(2, "Capt. Mike Rodriguez", "operations", 20, "intuitive", 0.8, 0.5, 0.6),
    HumanExpert(3, "Prof. Lisa Johnson", "service", 12, "collaborative", 0.75, 0.7, 0.9),
    HumanExpert(4, "Mr. David Kim", "risk", 18, "cautious", 0.7, 0.2, 0.7),
]

print("Fleet-sizing example data")
print("=" *  forty if False else "=" * 40)
print(f"Vehicle types: {len(vehicles)}")
for vehicle in vehicles:
    print(
        f"  {vehicle.name}: capacity {vehicle.capacity} cargo units, "
        f"acquisition cost ${vehicle.acquisition_cost:,.0f}, "
        f"reliability {vehicle.reliability:.0%}"
    )
print(f"Routes: {len(routes)}")
for route in routes:
    print(
        f"  {route.name}: demand {route.demand_units} units, "
        f"complexity {route.complexity:.1f}"
    )
print(f"Experts: {len(experts)}")
for expert in experts:
    print(f"  {expert.name}: {expert.expertise_area} expertise, {expert.decision_style} style")

In [ ]:
print("Human–AI partnership system setup")
print("=" * 42)
print("Components: rule-based AI proposal, expert adjustments, trust tracking, and feedback records.")

explainable_ai = ExplainableAI(vehicles, routes)
human_expertise = HumanExpertiseModel(experts)
trust_system = TrustDevelopmentSystem()

print(f"AI emphasis weights: {explainable_ai.reasoning_engine}")
print(f"Expertise areas represented: {len({expert.expertise_area for expert in experts})}")
print(f"Decision styles represented: {len({expert.decision_style for expert in experts})}")
print(f"Initial trust score: {0.5:.1%}")

In [ ]:
class CollaborativeDecisionEngine:
    """Coordinate proposal, expert review, synthesis, and feedback recording."""

    def __init__(
        self,
        ai_system: ExplainableAI,
        human_model: HumanExpertiseModel,
        trust_system: TrustDevelopmentSystem,
    ):
        self.ai_system = ai_system
        self.human_model = human_model
        self.trust_system = trust_system
        self.collaboration_history: List[CollaborativeDecision] = []
        self.performance_history: List[float] = []
        print("Collaborative decision engine initialized.")

    def make_collaborative_decision(
        self,
        context: Dict[str, Any],
        expert_id: int,
    ) -> CollaborativeDecision:
        """Run one complete human–AI interaction in the defined order."""
        ai_decision = self.ai_system.generate_decision(context)
        human_decision = self.human_model.get_human_decision(
            expert_id, ai_decision, context
        )
        final_allocation = self._synthesize_decision(ai_decision, human_decision)
        agreement = self._calculate_agreement(
            ai_decision.fleet_allocation,
            human_decision.fleet_allocation,
        )
        current_trust = (
            self.trust_system.trust_history[-1]
            if self.trust_system.trust_history
            else 0.5
        )
        performance = self._simulate_performance(final_allocation, context)
        outcome = self._determine_learning_outcome(performance, agreement)

        decision = CollaborativeDecision(
            timestamp=datetime.now(),
            ai_decision=ai_decision,
            human_input=human_decision,
            final_allocation=final_allocation,
            trust_score=current_trust,
            agreement_level=agreement,
            learning_outcome=outcome,
            performance_score=performance,
        )

        self.trust_system.update_trust(decision)
        expert = next(item for item in self.human_model.experts if item.id == expert_id)
        self.ai_system.learn_from_feedback(human_decision, performance, expert)
        self.collaboration_history.append(decision)
        self.performance_history.append(performance)
        return decision

    def _synthesize_decision(
        self,
        ai_decision: AIDecision,
        human_decision: HumanDecision,
    ) -> np.ndarray:
        """Blend allocations using current trust and human confidence."""
        trust_weight = (
            self.trust_system.trust_history[-1]
            if self.trust_system.trust_history
            else 0.5
        )
        human_confidence = human_decision.confidence
        total_weight = trust_weight + human_confidence
        ai_weight = trust_weight / total_weight
        human_weight = human_confidence / total_weight
        return (
            ai_weight * ai_decision.fleet_allocation
            + human_weight * human_decision.fleet_allocation
        )

    def _calculate_agreement(
        self,
        ai_allocation: np.ndarray,
        human_allocation: np.ndarray,
    ) -> float:
        """Compare normalized allocation patterns, not total fleet sizes."""
        total_ai = float(np.sum(ai_allocation))
        total_human = float(np.sum(human_allocation))
        if np.isclose(total_ai, 0.0) and np.isclose(total_human, 0.0):
            return 1.0
        if np.isclose(total_ai, 0.0) or np.isclose(total_human, 0.0):
            return 0.0
        ai_normalized = ai_allocation / total_ai
        human_normalized = human_allocation / total_human
        return float(np.clip(1.0 - np.mean(np.abs(ai_normalized - human_normalized)), 0.0, 1.0))

    def _simulate_performance(
        self,
        final_allocation: np.ndarray,
        context: Dict[str, Any],
    ) -> float:
        """Calculate a noisy demonstration score, not an observed outcome."""
        total_cost = self.ai_system._calculate_total_cost(final_allocation)
        demands = np.asarray(
            context.get("demands", [route.demand_units for route in self.ai_system.routes]),
            dtype=float,
        )
        service_level = self.ai_system._calculate_service_level(final_allocation, demands)
        cost_score = max(0.0, 1.0 - (total_cost - 320000.0) / 320000.0)
        score = 0.6 * service_level + 0.4 * cost_score
        score += float(np.random.normal(0.0, 0.05))
        return float(np.clip(score, 0.0, 1.0))

    def _determine_learning_outcome(self, performance: float, agreement: float) -> str:
        """Assign a descriptive label to this interaction's scores."""
        if performance > 0.8 and agreement > 0.8:
            return "High simulated performance and strong allocation agreement."
        if performance > 0.8 and agreement < 0.5:
            return "High simulated performance with low allocation agreement."
        if performance < 0.6 and agreement > 0.8:
            return "Strong allocation agreement with a low simulated score."
        if performance < 0.6 and agreement < 0.5:
            return "Low simulated performance and low allocation agreement."
        if agreement > 0.6:
            return "Moderate simulated performance with substantial agreement."
        return "Low allocation agreement; review the differing proposals."

    def get_partnership_metrics(self) -> Dict[str, float]:
        """Summarize recorded interactions and a simple early/recent comparison."""
        if not self.collaboration_history:
            return {
                "total_interactions": 0,
                "average_performance": 0.0,
                "average_agreement": 0.0,
                "trust_development": 0.5,
                "learning_effectiveness": 0.0,
            }

        performances = self.performance_history
        learning_effectiveness = 0.0
        if len(performances) > 10:
            early = float(np.mean(performances[:5]))
            recent = float(np.mean(performances[-5:]))
            if not np.isclose(early, 0.0):
                learning_effectiveness = (recent - early) / abs(early)

        return {
            "total_interactions": len(self.collaboration_history),
            "average_performance": float(np.mean(performances)),
            "average_agreement": float(
                np.mean([item.agreement_level for item in self.collaboration_history])
            ),
            "trust_development": self.trust_system.get_trust_metrics()["current_trust"],
            "learning_effectiveness": learning_effectiveness,
        }


collaborative_engine = CollaborativeDecisionEngine(
    explainable_ai,
    human_expertise,
    trust_system,
)
print(f"Experts available: {len(experts)}")
print("Decision synthesis: trust- and confidence-weighted average")

In [ ]:
def run_partnership_simulation(
    interactions: int = 50,
) -> List[CollaborativeDecision]:
    """Run interactions while rotating through expert profiles."""
    if interactions <= 0:
        raise ValueError("The number of interactions must be positive.")

    print("Starting human–AI partnership simulation")
    print(f"Interactions: {interactions}; experts: {len(experts)}")
    print("=" * 62)
    decisions = []

    for interaction in range(interactions):
        expert_id = experts[interaction % len(experts)].id
        context = {
            "demands": np.maximum(
                0.0,
                np.array([route.demand_units for route in routes])
                + np.random.normal(0.0, 2.0, len(routes)),
            ),
            "cost_focus": random.random() < 0.3,
            "service_focus": random.random() < 0.3,
            "uncertainty": 0.1 + random.random() * 0.1,
            "demand_uncertainty": random.random() * 0.2,
            "cost_volatility": random.random() * 0.2,
            "market_conditions": random.choice(["normal", "volatile", "growing"]),
            "seasonal_factors": random.random() < 0.2,
        }
        decision = collaborative_engine.make_collaborative_decision(context, expert_id)
        decisions.append(decision)

        report_interval = max(1, interactions // 10)
        if (interaction + 1) % report_interval == 0 or interaction + 1 == interactions:
            print(
                f"Interaction {interaction + 1:3d}/{interactions}: "
                f"trust before update {decision.trust_score:.1%}, "
                f"simulated performance {decision.performance_score:.1%}, "
                f"agreement {decision.agreement_level:.1%}"
            )

    print("Simulation complete.")
    return decisions


partnership_decisions = run_partnership_simulation(interactions=50)

In [ ]:
def visualize_partnership_results(
    decisions: List[CollaborativeDecision],
    experts: List[HumanExpert],
) -> None:
    """Plot trust, simulated performance, agreement, and expert summaries."""
    if not decisions:
        raise ValueError("At least one decision is required for visualization.")

    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    interaction_numbers = np.arange(1, len(decisions) + 1)
    trust_scores = [decision.trust_score for decision in decisions]
    performance_scores = [decision.performance_score for decision in decisions]
    agreement_scores = [decision.agreement_level for decision in decisions]

    ax1.plot(interaction_numbers, trust_scores, color="tab:blue", marker="o", markersize=3)
    ax1.axhline(0.8, color="tab:green", linestyle="--", label="High-trust reference")
    ax1.axhline(0.5, color="tab:orange", linestyle="--", label="Neutral starting score")
    ax1.set(title="Trust Before Each Interaction", xlabel="Interaction", ylabel="Trust score")
    ax1.set_ylim(0, 1)
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(interaction_numbers, performance_scores, color="tab:green", label="Simulated performance")
    ax2.plot(interaction_numbers, agreement_scores, color="tab:red", label="Allocation agreement")
    ax2.set(title="Simulated Performance and Agreement", xlabel="Interaction", ylabel="Score")
    ax2.set_ylim(0, 1)
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    expert_by_id = {expert.id: expert for expert in experts}
    expert_scores: Dict[int, List[float]] = {}
    for decision in decisions:
        if decision.human_input is not None:
            expert_id = decision.human_input.expert_id
            expert_scores.setdefault(expert_id, []).append(decision.performance_score)
    expert_ids = [expert.id for expert in experts if expert.id in expert_scores]
    expert_names = [expert_by_id[expert_id].name for expert_id in expert_ids]
    average_scores = [float(np.mean(expert_scores[expert_id])) for expert_id in expert_ids]
    bars = ax3.bar(expert_names, average_scores, color="steelblue", alpha=0.8)
    ax3.set(title="Mean Simulated Score by Expert", ylabel="Mean performance score")
    ax3.set_ylim(0, 1)
    ax3.tick_params(axis="x", rotation=35)
    ax3.grid(True, alpha=0.3, axis="y")
    for bar, score in zip(bars, average_scores):
        ax3.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f"{score:.3f}",
            ha="center",
            va="bottom",
        )

    window_size = 10
    moving_average = [
        float(np.mean(performance_scores[max(0, i - window_size + 1) : i + 1]))
        for i in range(len(performance_scores))
    ]
    ax4.plot(interaction_numbers, performance_scores, color="lightgray", label="Individual score")
    ax4.plot(
        interaction_numbers,
        moving_average,
        color="purple",
        linewidth=2,
        label=f"Trailing mean ({window_size} interactions)",
    )
    ax4.set(title="Simulated Score Across Interactions", xlabel="Interaction", ylabel="Performance score")
    ax4.set_ylim(0, 1)
    ax4.legend()
    ax4.grid(True, alpha=0.3)

    fig.suptitle("Human–AI Partnership Simulation", fontsize=16)
    fig.tight_layout()
    plt.show()


visualize_partnership_results(partnership_decisions, experts)

In [ ]:
def analyze_partnership_insights(
    decisions: List[CollaborativeDecision],
    engine: CollaborativeDecisionEngine,
    trust_tracker: TrustDevelopmentSystem,
    experts: List[HumanExpert],
) -> List[str]:
    """Summarize observed simulation metrics without inferring causality."""
    if not decisions:
        return ["There are no decisions to analyze."]

    insights = []
    trust_metrics = trust_tracker.get_trust_metrics()
    trust_direction = (
        "increased" if trust_metrics["trust_trend"] > 0.05
        else "decreased" if trust_metrics["trust_trend"] < -0.05
        else "changed little"
    )
    insights.append(
        f"Final trust score: {trust_metrics['current_trust']:.1%}; "
        f"the first-to-last change {trust_direction}."
    )

    performances = [decision.performance_score for decision in decisions]
    agreements = [decision.agreement_level for decision in decisions]
    insights.append(f"Mean simulated performance score: {np.mean(performances):.1%}.")
    insights.append(f"Mean normalized allocation agreement: {np.mean(agreements):.1%}.")

    metrics = engine.get_partnership_metrics()
    insights.append(
        "Early-to-recent performance change indicator: "
        f"{metrics['learning_effectiveness']:+.1%}. This is descriptive, not proof of learning."
    )

    expert_by_id = {expert.id: expert for expert in experts}
    expert_scores: Dict[int, List[float]] = {}
    for decision in decisions:
        if decision.human_input is not None:
            expert_id = decision.human_input.expert_id
            expert_scores.setdefault(expert_id, []).append(decision.performance_score)
    if expert_scores:
        top_id = max(expert_scores, key=lambda key: np.mean(expert_scores[key]))
        insights.append(
            f"Highest mean simulated score among these expert groups: "
            f"{expert_by_id[top_id].name} ({np.mean(expert_scores[top_id]):.1%}). "
            "This descriptive comparison does not establish an expert effect."
        )

    insights.append(
        f"Feedback records meeting the success threshold: "
        f"{len(engine.ai_system.success_patterns)}. The allocation rule is not retrained."
    )
    insights.extend(trust_tracker.get_trust_insights())
    return insights


insights = analyze_partnership_insights(
    partnership_decisions,
    collaborative_engine,
    trust_system,
    experts,
)
print("Partnership simulation insights")
print("=" * 34)
for insight in insights:
    print(f"- {insight}")

In [ ]:
if not partnership_decisions:
    raise ValueError("The simulation must produce decisions before results are summarized.")

final_decision = partnership_decisions[-1]
final_allocation = final_decision.final_allocation
optimal_fleet = {
    (vehicle.id, route.id): float(final_allocation[i, j])
    for i, vehicle in enumerate(vehicles)
    for j, route in enumerate(routes)
    if final_allocation[i, j] > 0
}

# Compute cost from the final collaborative allocation, not from the AI proposal.
final_total_cost = explainable_ai._calculate_total_cost(final_allocation)
trust_development = list(trust_system.trust_history)
partnership_metrics = collaborative_engine.get_partnership_metrics()

# The AI-only comparator is not simulated, so this comparison is deliberately undefined.
decision_quality_improvement = float("nan")
partnership_benefits = {
    "trust_change_from_neutral": trust_development[-1] - 0.5,
    "mean_allocation_agreement": partnership_metrics["average_agreement"],
    "early_to_recent_performance_change": partnership_metrics["learning_effectiveness"],
    "mean_ai_confidence": float(
        np.mean([decision.ai_decision.confidence for decision in partnership_decisions])
    ),
}

partnership_result = PartnershipResult(
    optimal_fleet=optimal_fleet,
    total_cost=final_total_cost,
    trust_development=trust_development,
    decision_quality_improvement=decision_quality_improvement,
    learning_effectiveness=partnership_metrics["learning_effectiveness"],
    partnership_benefits=partnership_benefits,
    computation_time=float("nan"),
    insights=insights,
)

print("Final collaborative allocation")
print("=" * 34)
for (vehicle_id, route_id), count in partnership_result.optimal_fleet.items():
    vehicle = next(item for item in vehicles if item.id == vehicle_id)
    route = next(item for item in routes if item.id == route_id)
    print(f"  {count:.2f} assigned units of {vehicle.name} to {route.name}")
print(f"Modeled cost of final allocation: ${partnership_result.total_cost:,.2f}")
print(f"Final trust score: {trust_development[-1]:.1%}")
print(f"Interactions: {len(partnership_decisions)}")
print("AI-only quality improvement: not calculated; no AI-only outcome series is modeled.")
print("Elapsed computation time: not measured.")

In [ ]:
def comprehensive_partnership_comparison() -> None:
    """Report this simulation's results and identify unavailable comparisons."""
    metrics = collaborative_engine.get_partnership_metrics()
    print("Partnership simulation summary")
    print("=" * 34)
    print(f"Interactions recorded: {metrics['total_interactions']}")
    print(f"Mean simulated performance score: {metrics['average_performance']:.1%}")
    print(f"Mean normalized allocation agreement: {metrics['average_agreement']:.1%}")
    print(f"Final trust score: {metrics['trust_development']:.1%}")
    print(
        "Early-to-recent performance change indicator: "
        f"{metrics['learning_effectiveness']:+.1%}"
    )
    print(f"Final collaborative allocation cost: ${partnership_result.total_cost:,.2f}")
    print(
        "No cross-method cost, runtime, or quality comparison is calculated here: "
        "comparable outcomes for other methods are not generated by this simulation."
    )
    print(
        "Interpretation: these metrics describe the specified rule-based simulation; "
        "they do not establish optimality, causal improvement, or operational performance."
    )


comprehensive_partnership_comparison()

## Interpretation and limitations

The plots summarize the simulated interaction sequence. Trust plotted at interaction \(k\) is the score available before that interaction; the trust system appends the updated score after the interaction. Performance is a noisy score calculated from modeled capacity coverage and a fixed cost normalization. Agreement compares the relative allocation patterns after normalizing each allocation by its total. These quantities answer different questions and should not be treated as interchangeable measures of decision quality.

The expert comparison is descriptive: experts are evaluated on the outcomes generated by this particular simulation, not in a controlled experiment. Randomly varying contexts and performance noise can affect these averages. The moving average is a summary of nearby scores, not evidence that the AI policy learned. The feedback mechanism stores successful examples and running preference summaries, but the fixed proposal rule does not use those records to change future allocations.

This example deliberately omits integer fleet constraints, a formal fleet-cost optimization objective, vehicle availability across routes, route scheduling, stochastic service reliability, and a separately measured AI-only baseline. Consequently, the final allocation is a fractional planning signal and the cost is a model calculation, not a verified operating plan. A real deployment would require validated data, explicit operational constraints, and evaluation against comparable baselines.

In [ ]:
print("Tier 7 summary: Human–AI Symbiotic Partnership")
print("=" * 54)
print("Implemented components:")
print("  - Rule-based AI proposals with contextual explanations")
print("  - Expert profiles and specialty-based allocation adjustments")
print("  - Trust scoring based on simulated interaction indicators")
print("  - Weighted synthesis of AI and human allocation matrices")
print("  - Feedback records for successful human decisions")
print()
print("Observed in this run:")
print(f"  - Interactions: {len(partnership_decisions)}")
print(f"  - Final trust score: {trust_system.get_trust_metrics()['current_trust']:.1%}")
print(f"  - Mean simulated performance: {partnership_metrics['average_performance']:.1%}")
print(f"  - Mean allocation agreement: {partnership_metrics['average_agreement']:.1%}")
print(f"  - Final allocation cost under the stated cost rule: ${partnership_result.total_cost:,.2f}")
print()
print("Interpretation boundary:")
print("  These are outcomes of an illustrative simulation, not measured operational results.")
print("  The rule-based proposal is not an optimization algorithm, and feedback does not retrain it.")
print("  No AI-only baseline or cross-method performance comparison is available in this run.")